In [ ]:
# =============================================================================
# Cell 1: Mount Google Drive & Install Dependencies
# =============================================================================
!pip install -q open3d plyfile scikit-learn pandas numpy matplotlib seaborn tqdm

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# =============================================================================
# Cell 2: Configuration & Paths (ZIP from Drive, Extract to Session, Save to Drive)
# =============================================================================
import os
import numpy as np
import torch
import open3d as o3d
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from tqdm import tqdm
import json
import time
import zipfile
import warnings
warnings.filterwarnings('ignore')

# ---- Mount Google Drive (zip is stored here) --------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ---- Step 1: Define paths ----------------------------------------------------
# The zip file is ALREADY on Drive
ZIP_NAME = "segmentation_jetcobot_internship-20260818T202713Z-1-001.zip"
ZIP_PATH = os.path.join("/content/drive/MyDrive/segmentation_jetcobot_internship", ZIP_NAME)

# Where to extract the zip (local Colab session for fast I/O)
EXTRACT_PATH = "/content/segmentation_jetcobot_internship"

# Output base (Google Drive, persistent storage)
OUTPUT_BASE_DIR = "/content/drive/MyDrive/segmentation_jetcobot_internship"

# ---- Step 2: Check if zip exists on Drive ------------------------------------
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"Zip file not found at {ZIP_PATH}. "
        "Please make sure the zip is in your Drive at: "
        "/content/drive/MyDrive/segmentation_jetcobot_internship/"
    )

# ---- Step 3: Extract zip to local session (if not already extracted) --------
if not os.path.exists(EXTRACT_PATH):
    print(f"📦 Extracting {ZIP_NAME} from Drive to local session (/content/) ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall("/content")
    print(f"✅ Extraction complete. Source data at: {EXTRACT_PATH}")
else:
    print(f"✅ Source data already extracted at: {EXTRACT_PATH}")

# ---- Step 4: Define SOURCE base (read from extracted local session) ---------
SOURCE_BASE_DIR = EXTRACT_PATH

# ---- Step 5: Set all input paths (from SOURCE) ------------------------------
PLANT_NAME = "Ribes_04"   # change as needed

# GT reconstruction
GT_DIR = os.path.join(SOURCE_BASE_DIR, "gt_reconstruction")
PLY_PATH = os.path.join(GT_DIR, f"{PLANT_NAME}.ply")
SEMANTIC_PATH = os.path.join(GT_DIR, f"{PLANT_NAME}_SemanticLabels.txt")
INSTANCE_PATH = os.path.join(GT_DIR, f"{PLANT_NAME}_InstanceLabels.txt")
CONFIDENCE_PATH = os.path.join(GT_DIR, f"{PLANT_NAME}_Confidence.txt")

# Noisy variants (manifest)
NOISY_DIR = os.path.join(SOURCE_BASE_DIR, "noisy_reconstruction")
DEGRADED_DIR = os.path.join(NOISY_DIR, "degraded")
MANIFEST_PATH = os.path.join(DEGRADED_DIR, "manifest.csv")

# Existing PTv3 supervised checkpoint (inside the extracted source)
PTV3_ROBUST_DIR = os.path.join(SOURCE_BASE_DIR, "outputs", PLANT_NAME, "robustness_eval_ptv3")
PTV3_CKPT_PATH = os.path.join(PTV3_ROBUST_DIR, "checkpoints", "ptv3_best.pth")

# ---- Step 6: Set all OUTPUT paths (to Google Drive) -------------------------
OUTPUT_DIR = os.path.join(OUTPUT_BASE_DIR, "outputs", PLANT_NAME, "ssl_bl")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

METRICS_DIR = os.path.join(OUTPUT_DIR, "tables")
os.makedirs(METRICS_DIR, exist_ok=True)

FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

REPORT_DIR = os.path.join(OUTPUT_DIR, "report")
os.makedirs(REPORT_DIR, exist_ok=True)

CM_DIR = os.path.join(METRICS_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

# ---- Step 7: Semantic class legend ------------------------------------------
SEMANTIC_CLASS_NAMES = {
    0: "Background / Pot / Soil",
    1: "Stem",
    2: "Leaf",
}
NUM_CLASSES = len(SEMANTIC_CLASS_NAMES)

# ---- Step 8: Random seeds ---------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ---- Step 9: Print summary for confirmation --------------------------------
print("\n" + "="*60)
print("✅ Configuration Summary:")
print(f"  Zip location (Drive): {ZIP_PATH}")
print(f"  Source data (read from local session): {SOURCE_BASE_DIR}")
print(f"    - GT folder: {GT_DIR}")
print(f"    - Manifest:  {MANIFEST_PATH}")
print(f"    - PTv3 checkpoint exists: {os.path.exists(PTV3_CKPT_PATH)}")
print(f"  Outputs (saved to Google Drive): {OUTPUT_BASE_DIR}")
print(f"    - Checkpoints -> {CHECKPOINT_DIR}")
print(f"    - Metrics     -> {METRICS_DIR}")
print(f"    - Figures     -> {FIG_DIR}")
print(f"    - Reports     -> {REPORT_DIR}")
print("="*60 + "\n")

In [ ]:

# =============================================================================
# Cell 3: Hyperparameters
# =============================================================================
# ---- Data sampling ---------------------------------------------------------
N_POINTS_PER_SAMPLE = 4096   # per training crop (as in paper)
USE_RGB = True               # if GT .ply has color

# ---- Barlow Twins SSL ------------------------------------------------------
SSL_EPOCHS = 300
SSL_BATCH_SIZE = 32          # paper uses 150, but adjust for memory
SSL_LEARNING_RATE = 1e-4
SSL_LAMBDA = 5e-3            # weight of off-diagonal term
SSL_WARMUP_EPOCHS = 10
SSL_N_SAMPLES_PER_EPOCH = 500   # number of crops per epoch

# ---- Encoder architecture (same as PTv3-lite, for fair comparison) ---------
STAGE_CHANNELS = (64, 128, 256, 512)
STAGE_DEPTHS = (2, 2, 2, 2)
NUM_HEADS = 4
PATCH_SIZE = 32
POOL_RATIO = 4
GRID_RES = 1024
DROPOUT = 0.1

# ---- Linear probe (head) training ------------------------------------------
PROBE_EPOCHS = 40
PROBE_BATCH_SIZE = 8
PROBE_LEARNING_RATE = 1e-3
PROBE_WEIGHT_DECAY = 1e-4
PROBE_LR_DECAY_STEP = 15
PROBE_LR_DECAY_GAMMA = 0.5

# ---- Train/val split (point-level) -----------------------------------------
TRAIN_POINT_FRACTION = 0.85
N_TRAIN_SAMPLES_PER_EPOCH = 300
N_VAL_SAMPLES_PER_EPOCH = 60

# ---- Whole-cloud evaluation ------------------------------------------------
N_REPEATS_EVAL = 4

# ---- Projector dimensions (Barlow Twins) -----------------------------------
PROJECTOR_HIDDEN = 512
PROJECTOR_OUT = 128

print(f"N_POINTS_PER_SAMPLE = {N_POINTS_PER_SAMPLE}, USE_RGB = {USE_RGB}")
print(f"SSL_EPOCHS = {SSL_EPOCHS}, SSL_LAMBDA = {SSL_LAMBDA}")
print(f"Probe epochs = {PROBE_EPOCHS}")



In [ ]:
# =============================================================================
# Cell 4: Load GT point cloud and labels
# =============================================================================
def load_point_cloud(path):
    pcd = o3d.io.read_point_cloud(path)
    pts = np.asarray(pcd.points, dtype=np.float64)
    cols = np.asarray(pcd.colors, dtype=np.float64) if pcd.has_colors() else None
    return pts, cols

def load_label_file(path, dtype=np.int64):
    return np.loadtxt(path, dtype=dtype).reshape(-1)

gt_points, gt_rgb = load_point_cloud(PLY_PATH)
gt_semantic = load_label_file(SEMANTIC_PATH, dtype=np.int64)
gt_instance = load_label_file(INSTANCE_PATH, dtype=np.int64)
gt_confidence = load_label_file(CONFIDENCE_PATH, dtype=np.float64)

N_GT = gt_points.shape[0]
assert N_GT == gt_semantic.shape[0] == gt_instance.shape[0] == gt_confidence.shape[0]

if gt_rgb is None or not USE_RGB:
    USE_RGB = False
    ADDITIONAL_CHANNELS = 0
    print("RGB not available or disabled. Using only XYZ.")
else:
    ADDITIONAL_CHANNELS = 3
    print("Using RGB as additional features.")

# Normalization frame (shared with degraded variants)
GLOBAL_CENTER = gt_points.mean(axis=0)
bbox_min, bbox_max = gt_points.min(axis=0), gt_points.max(axis=0)
CHAR_SIZE = float(np.linalg.norm(bbox_max - bbox_min))

def normalize_xyz(points):
    return (points - GLOBAL_CENTER) / CHAR_SIZE

def make_features(rgb):
    if not USE_RGB:
        return None
    return rgb.astype(np.float64) if rgb is not None else np.zeros((0, 3))

gt_xyz_norm = normalize_xyz(gt_points)
gt_features = make_features(gt_rgb)  # [N,3] or None

unique_sem, counts = np.unique(gt_semantic, return_counts=True)
print(f"GT points: {N_GT:,}")
print("Class distribution:")
for u, c in zip(unique_sem, counts):
    print(f"  {u}: {SEMANTIC_CLASS_NAMES.get(int(u), 'unknown')} -> {c:,} ({100*c/N_GT:.2f}%)")


In [ ]:
# =============================================================================
# Cell 5: Shared building blocks (encoder, serialized attention, etc.)
# =============================================================================
import torch.nn as nn
import torch.nn.functional as F

# ---- Morton ordering helpers (copied from Utonia notebook) -----------------
def morton_code(xyz_int, bits=10):
    x, y, z = xyz_int[..., 0].long(), xyz_int[..., 1].long(), xyz_int[..., 2].long()
    def spread_bits(v):
        v = v & ((1 << bits) - 1)
        result = torch.zeros_like(v)
        for i in range(bits):
            result |= ((v >> i) & 1) << (3 * i)
        return result
    return (spread_bits(x) << 2) | (spread_bits(y) << 1) | spread_bits(z)

def serialize_order(xyz, grid_res):
    B, N, _ = xyz.shape
    xyz_min = xyz.amin(dim=1, keepdim=True)
    xyz_max = xyz.amax(dim=1, keepdim=True)
    xyz_norm01 = (xyz - xyz_min) / (xyz_max - xyz_min + 1e-8)
    bits = max(1, int(np.log2(grid_res)))
    xyz_int = (xyz_norm01 * (grid_res - 1)).round().clamp(0, grid_res - 1)
    codes = morton_code(xyz_int, bits=bits)
    order = torch.argsort(codes, dim=1)
    return order

def gather_seq(x, order):
    B, N, C_ = x.shape
    idx = order.unsqueeze(-1).expand(-1, -1, C_)
    return torch.gather(x, 1, idx)

def scatter_seq(x_sorted, order):
    inv = torch.argsort(order, dim=1)
    return gather_seq(x_sorted, inv)

def square_distance(src, dst):
    return torch.sum((src[:, :, None, :] - dst[:, None, :, :]) ** 2, dim=-1)

def index_points(points, idx):
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape); view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape); repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long, device=device).view(view_shape).repeat(repeat_shape)
    return points[batch_indices, idx, :]

# ---- SerializedAttentionStage, SerializedPool, FeaturePropagation ----------
class SerializedAttentionStage(nn.Module):
    def __init__(self, channels, depth, num_heads, patch_size, mlp_ratio=2, dropout=0.0):
        super().__init__()
        self.patch_size = patch_size
        self.pos_mlp = nn.Sequential(nn.Linear(3, channels), nn.GELU(), nn.Linear(channels, channels))
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=channels, nhead=num_heads, dim_feedforward=channels * mlp_ratio,
                dropout=dropout, batch_first=True, norm_first=True, activation="gelu",
            ) for _ in range(depth)
        ])

    def forward(self, x, xyz):
        B, N, C_ = x.shape
        P = self.patch_size if (N % self.patch_size == 0 and N >= self.patch_size) else N
        num_patches = N // P
        xyz_p = xyz.view(B, num_patches, P, 3)
        centroid = xyz_p.mean(dim=2, keepdim=True)
        rel_pos = (xyz_p - centroid).reshape(B, N, 3)
        x = x + self.pos_mlp(rel_pos)
        x_p = x.view(B * num_patches, P, C_)
        for blk in self.blocks:
            x_p = blk(x_p)
        return x_p.view(B, N, C_)

class SerializedPool(nn.Module):
    def __init__(self, in_ch, out_ch, ratio):
        super().__init__()
        self.ratio = ratio
        self.proj = nn.Sequential(nn.Linear(in_ch, out_ch), nn.GELU())

    def forward(self, x, xyz):
        B, N, C_ = x.shape
        r = self.ratio if N % self.ratio == 0 else 1
        if r == 1:
            return self.proj(x), xyz
        x = x.view(B, N // r, r, C_).max(dim=2)[0]
        xyz = xyz.view(B, N // r, r, 3).mean(dim=2)
        return self.proj(x), xyz

class FeaturePropagation(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_ch, out_ch), nn.GELU(),
            nn.Linear(out_ch, out_ch), nn.GELU(),
        )

    def forward(self, xyz1, xyz2, feat1, feat2):
        B, N1, _ = xyz1.shape
        N2 = xyz2.shape[1]
        if N2 == 1:
            interp = feat2.repeat(1, N1, 1)
        else:
            dists = square_distance(xyz1, xyz2)
            dists, idx = dists.sort(dim=-1)
            dists, idx = dists[:, :, :3], idx[:, :, :3]
            dist_recip = 1.0 / (dists + 1e-8)
            norm = dist_recip.sum(dim=2, keepdim=True)
            weight = dist_recip / norm
            interp = torch.sum(index_points(feat2, idx) * weight.unsqueeze(-1), dim=2)
        x = torch.cat([feat1, interp], dim=-1) if feat1 is not None else interp
        return self.mlp(x)

# ---- UtoniaEncoderLite (same as before) ------------------------------------
class UtoniaEncoderLite(nn.Module):
    def __init__(self, additional_channel=3, stage_channels=(64,128,256,512),
                 stage_depths=(2,2,2,2), num_heads=4, patch_size=32, pool_ratio=4,
                 grid_res=1024, dropout=0.1):
        super().__init__()
        in_ch = 3 + additional_channel
        self.grid_res = grid_res
        C0, C1, C2, C3 = stage_channels
        self.input_proj = nn.Sequential(nn.Linear(in_ch, C0), nn.GELU())
        self.mask_token = nn.Parameter(torch.zeros(1, 1, in_ch))  # not used for BT, kept for compatibility
        nn.init.normal_(self.mask_token, std=0.02)

        self.stage0 = SerializedAttentionStage(C0, stage_depths[0], num_heads, patch_size, dropout=dropout)
        self.pool0 = SerializedPool(C0, C1, pool_ratio)
        self.stage1 = SerializedAttentionStage(C1, stage_depths[1], num_heads, patch_size, dropout=dropout)
        self.pool1 = SerializedPool(C1, C2, pool_ratio)
        self.stage2 = SerializedAttentionStage(C2, stage_depths[2], num_heads, patch_size, dropout=dropout)
        self.pool2 = SerializedPool(C2, C3, pool_ratio)
        self.stage3 = SerializedAttentionStage(C3, stage_depths[3], num_heads, patch_size, dropout=dropout)

    def forward(self, xyz_bcn, features_bcn=None, mask_bn=None):
        xyz = xyz_bcn.transpose(1, 2).contiguous()
        if features_bcn is not None and features_bcn.shape[1] > 0:
            feat = features_bcn.transpose(1, 2).contiguous()
            x_in = torch.cat([xyz, feat], dim=-1)
        else:
            x_in = xyz

        if mask_bn is not None:
            mask = mask_bn.unsqueeze(-1).to(x_in.dtype)
            x_in = x_in * (1 - mask) + self.mask_token * mask

        order0 = serialize_order(xyz, self.grid_res)
        xyz0 = gather_seq(xyz, order0)
        x_in = gather_seq(x_in, order0)

        x0 = self.input_proj(x_in)
        x0 = self.stage0(x0, xyz0)

        x1, xyz1 = self.pool0(x0, xyz0)
        x1 = self.stage1(x1, xyz1)

        x2, xyz2 = self.pool1(x1, xyz1)
        x2 = self.stage2(x2, xyz2)

        x3, xyz3 = self.pool2(x2, xyz2)
        x3 = self.stage3(x3, xyz3)

        feats = {
            "x0": x0, "xyz0": xyz0,
            "x1": x1, "xyz1": xyz1,
            "x2": x2, "xyz2": xyz2,
            "x3": x3, "xyz3": xyz3,
            "order0": order0,
        }
        return feats

# ---- Barlow Twins Projector (CORRECTED) ------------------------------------
class Projector(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, output_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        # x is already globally pooled: shape [B, input_dim]
        return self.net(x)

# ---- Barlow Twins loss -----------------------------------------------------
def barlow_twins_loss(z1, z2, lambda_coeff=5e-3):
    """
    z1, z2: [B, D] embeddings (batch_size, projector_out)
    """
    B, D = z1.shape
    # normalize along batch dimension
    z1_norm = (z1 - z1.mean(dim=0)) / z1.std(dim=0)
    z2_norm = (z2 - z2.mean(dim=0)) / z2.std(dim=0)
    # cross-correlation matrix
    c = torch.matmul(z1_norm.T, z2_norm) / B  # [D, D]
    on_diag = torch.diagonal(c).add_(-1).pow_(2).sum()
    off_diag = (c * (1 - torch.eye(D, device=c.device))).pow_(2).sum()
    loss = on_diag + lambda_coeff * off_diag
    return loss

print("Encoder, projector (fixed), and Barlow Twins loss defined.")

In [ ]:

# =============================================================================
# Cell 6: Dataset for SSL (unlabeled crops with two augmentations)
# =============================================================================
import torch.utils.data as data

class SSLPointCloudDataset(data.Dataset):
    """Samples random crops from the full GT cloud, applies two different augmentations."""
    def __init__(self, points_norm, features, n_points, samples_per_epoch,
                 use_rgb=True, seed=0):
        self.points_norm = points_norm
        self.features = features  # [N,3] or None
        self.n_points = n_points
        self.samples_per_epoch = samples_per_epoch
        self.use_rgb = use_rgb
        self.rng = np.random.default_rng(seed)
        self.N = points_norm.shape[0]

    def __len__(self):
        return self.samples_per_epoch

    def __getitem__(self, idx):
        # Sample random points
        replace = self.N < self.n_points
        sel = self.rng.choice(self.N, size=self.n_points, replace=replace)
        pts = self.points_norm[sel].copy()
        if self.features is not None:
            feat = self.features[sel].copy()
        else:
            feat = np.zeros((self.n_points, 0))

        # Generate two random augmentations (views)
        pts1, feat1 = self._augment(pts, feat)
        pts2, feat2 = self._augment(pts, feat)

        # Convert to torch tensors
        pts1_t = torch.from_numpy(pts1.astype(np.float32)).transpose(0, 1)  # [3, N]
        feat1_t = torch.from_numpy(feat1.astype(np.float32)).transpose(0, 1) if feat1.shape[1] > 0 else None
        pts2_t = torch.from_numpy(pts2.astype(np.float32)).transpose(0, 1)
        feat2_t = torch.from_numpy(feat2.astype(np.float32)).transpose(0, 1) if feat2.shape[1] > 0 else None

        return pts1_t, feat1_t, pts2_t, feat2_t

    def _augment(self, pts, feat):
        # Geometric augmentations (rotation, scaling, jitter)
        # Random rotation around z-axis
        theta = self.rng.uniform(0, 2 * np.pi)
        c, s = np.cos(theta), np.sin(theta)
        R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float64)
        pts_aug = pts @ R.T

        # Random scaling
        scale = self.rng.uniform(0.9, 1.1)
        pts_aug = pts_aug * scale

        # Random jitter (small coordinate noise)
        pts_aug = pts_aug + self.rng.normal(0, 0.005, size=pts_aug.shape)

        # Photometric augmentations (color jitter) if RGB available
        feat_aug = feat.copy()
        if self.use_rgb and feat.shape[1] >= 3:
            # Color jitter: brightness, contrast, saturation, hue
            # We'll implement simple brightness and contrast
            brightness = self.rng.uniform(0.8, 1.2)
            contrast = self.rng.uniform(0.8, 1.2)
            feat_aug[:, :3] = feat_aug[:, :3] * contrast + brightness - 0.5
            # Clip to [0,1]
            feat_aug[:, :3] = np.clip(feat_aug[:, :3], 0, 1)
        return pts_aug, feat_aug

# Create SSL dataset and dataloader
ssl_dataset = SSLPointCloudDataset(
    gt_xyz_norm, gt_features, N_POINTS_PER_SAMPLE,
    SSL_N_SAMPLES_PER_EPOCH, use_rgb=USE_RGB, seed=RANDOM_SEED
)
ssl_loader = data.DataLoader(ssl_dataset, batch_size=SSL_BATCH_SIZE,
                             shuffle=False, num_workers=0, drop_last=True)

print(f"SSL dataset size: {len(ssl_dataset)} samples per epoch, batch size {SSL_BATCH_SIZE}")




In [ ]:
# =============================================================================
# Cell 7: Dataset for supervised probe training (labeled crops)
# =============================================================================
from torch.utils.data import DataLoader

# Point-level train/val split (same as before)
rng_split = np.random.default_rng(RANDOM_SEED)
perm = rng_split.permutation(N_GT)
n_train = int(round(N_GT * TRAIN_POINT_FRACTION))
train_pool = perm[:n_train]
val_pool = perm[n_train:]

class LabeledPatchDataset(data.Dataset):
    def __init__(self, points_norm, features, labels, index_pool, n_points, samples_per_epoch,
                 augment=True, seed=0):
        self.points_norm = points_norm
        self.features = features
        self.labels = labels
        self.index_pool = index_pool
        self.n_points = n_points
        self.samples_per_epoch = samples_per_epoch
        self.augment = augment
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self.samples_per_epoch

    def __getitem__(self, _):
        replace = len(self.index_pool) < self.n_points
        sel = self.rng.choice(self.index_pool, size=self.n_points, replace=replace)
        pts = self.points_norm[sel].copy()
        lbl = self.labels[sel].copy()
        feat = self.features[sel].copy() if self.features is not None else np.zeros((self.n_points, 0))

        if self.augment:
            theta = self.rng.uniform(0, 2 * np.pi)
            c, s = np.cos(theta), np.sin(theta)
            R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float64)
            pts = pts @ R.T
            pts = pts + self.rng.normal(0, 0.005, size=pts.shape)
            pts = pts * self.rng.uniform(0.9, 1.1)

        pts_t = torch.from_numpy(pts.astype(np.float32)).transpose(0, 1)
        feat_t = torch.from_numpy(feat.astype(np.float32)).transpose(0, 1) if feat.shape[1] > 0 else None
        lbl_t = torch.from_numpy(lbl.astype(np.int64))
        return pts_t, feat_t, lbl_t

train_ds = LabeledPatchDataset(gt_xyz_norm, gt_features, gt_semantic, train_pool,
                               N_POINTS_PER_SAMPLE, N_TRAIN_SAMPLES_PER_EPOCH, augment=True, seed=RANDOM_SEED)
val_ds = LabeledPatchDataset(gt_xyz_norm, gt_features, gt_semantic, val_pool,
                             N_POINTS_PER_SAMPLE, N_VAL_SAMPLES_PER_EPOCH, augment=False, seed=RANDOM_SEED+1)

train_loader = DataLoader(train_ds, batch_size=PROBE_BATCH_SIZE, shuffle=False, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=PROBE_BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train point pool: {len(train_pool):,}, Val pool: {len(val_pool):,}")

In [ ]:
# =============================================================================
# Cell 8: Class weighting (inverse frequency)
# =============================================================================
class_counts_full = np.zeros(NUM_CLASSES, dtype=np.float64)
for u, c in zip(unique_sem, counts):
    if 0 <= int(u) < NUM_CLASSES:
        class_counts_full[int(u)] = c
class_counts_full = np.clip(class_counts_full, 1, None)
class_weights = 1.0 / class_counts_full
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights_t = torch.tensor(class_weights, dtype=torch.float32)
print("Class weights:", class_weights)


In [ ]:
# =============================================================================
# Cell 9: Instantiate encoder and projector (auto-detect output dim)
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

encoder = UtoniaEncoderLite(
    additional_channel=ADDITIONAL_CHANNELS,
    stage_channels=STAGE_CHANNELS,
    stage_depths=STAGE_DEPTHS,
    num_heads=NUM_HEADS,
    patch_size=PATCH_SIZE,
    pool_ratio=POOL_RATIO,
    grid_res=GRID_RES,
    dropout=DROPOUT,
).to(device)

# ---- Detect actual output dimension -----------------------------------------
with torch.no_grad():
    dummy_pts = torch.randn(2, 3, N_POINTS_PER_SAMPLE).to(device)  # [B,3,N]
    dummy_feats = torch.randn(2, ADDITIONAL_CHANNELS, N_POINTS_PER_SAMPLE).to(device) if ADDITIONAL_CHANNELS > 0 else None
    dummy_out = encoder(dummy_pts, dummy_feats, mask_bn=None)
    actual_feat_dim = dummy_out["x3"].shape[-1]   # [B, N, C] -> C
    print(f"Encoder output feature dimension: {actual_feat_dim}")

# ---- Instantiate projector with the actual dimension -------------------------
projector = Projector(input_dim=actual_feat_dim, hidden_dim=PROJECTOR_HIDDEN, output_dim=PROJECTOR_OUT).to(device)

# Optimizer
ssl_optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(projector.parameters()),
    lr=SSL_LEARNING_RATE,
    weight_decay=1e-6
)

# Warmup scheduler (same as before)
def warmup_lr_scheduler(optimizer, warmup_iters, warmup_factor):
    def f(x):
        if x >= warmup_iters:
            return 1
        alpha = float(x) / warmup_iters
        return warmup_factor * (1 - alpha) + alpha
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=f)

warmup_iters = SSL_WARMUP_EPOCHS * len(ssl_loader)
warmup_scheduler = warmup_lr_scheduler(ssl_optimizer, warmup_iters, 0.1)

print("Encoder and projector ready.")

In [ ]:
# =============================================================================
# Cell 10: Barlow Twins pretraining
# =============================================================================
ssl_history = {"epoch": [], "loss": []}
best_ssl_loss = float('inf')
ssl_encoder_path = os.path.join(CHECKPOINT_DIR, "barlow_encoder_best.pth")

t0 = time.time()
for epoch in range(1, SSL_EPOCHS + 1):
    encoder.train()
    projector.train()
    total_loss = 0.0
    for batch in tqdm(ssl_loader, desc=f"SSL Epoch {epoch}/{SSL_EPOCHS}", leave=False):
        pts1, feat1, pts2, feat2 = batch
        pts1, pts2 = pts1.to(device), pts2.to(device)
        if feat1 is not None:
            feat1, feat2 = feat1.to(device), feat2.to(device)
        else:
            feat1 = feat2 = None

        # Forward through encoder
        feats1 = encoder(pts1, feat1, mask_bn=None)
        feats2 = encoder(pts2, feat2, mask_bn=None)

        # Get final stage features (x3) and global pool
        z1 = feats1["x3"].max(dim=1)[0]  # [B, C3]
        z2 = feats2["x3"].max(dim=1)[0]

        # Project
        z1 = projector(z1)
        z2 = projector(z2)

        # Barlow Twins loss
        loss = barlow_twins_loss(z1, z2, lambda_coeff=SSL_LAMBDA)

        ssl_optimizer.zero_grad()
        loss.backward()
        ssl_optimizer.step()
        warmup_scheduler.step()

        total_loss += loss.item() * pts1.size(0)

    avg_loss = total_loss / len(ssl_dataset)
    ssl_history["epoch"].append(epoch)
    ssl_history["loss"].append(avg_loss)

    if avg_loss < best_ssl_loss:
        best_ssl_loss = avg_loss
        torch.save({
            "encoder_state_dict": encoder.state_dict(),
            "projector_state_dict": projector.state_dict(),
            "epoch": epoch,
            "loss": avg_loss,
            "additional_channel": ADDITIONAL_CHANNELS,
            "stage_channels": STAGE_CHANNELS,
            "stage_depths": STAGE_DEPTHS,
            "num_heads": NUM_HEADS,
            "patch_size": PATCH_SIZE,
            "pool_ratio": POOL_RATIO,
            "grid_res": GRID_RES,
        }, ssl_encoder_path)

    if epoch == 1 or epoch % 10 == 0 or epoch == SSL_EPOCHS:
        elapsed = time.time() - t0
        print(f"Epoch {epoch:3d}/{SSL_EPOCHS} | SSL loss: {avg_loss:.6f} | best: {best_ssl_loss:.6f} | {elapsed:.1f}s")

print("SSL pretraining completed. Best encoder saved.")

# Save loss history
pd.DataFrame(ssl_history).to_csv(os.path.join(METRICS_DIR, "ssl_pretrain_history.csv"), index=False)

In [ ]:
print(f"z1 shape: {z1.shape}")   # Should be [B, 512] ideally

In [ ]:
# =============================================================================
# Cell 10.5: Metric utilities (must be defined BEFORE probe training in Cell 11)
# =============================================================================
def compute_metrics(y_true, y_pred, num_classes=NUM_CLASSES):
    labels = list(range(num_classes))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    iou_per_class = np.full(num_classes, np.nan)
    for c in range(num_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        denom = tp + fp + fn
        if denom > 0:
            iou_per_class[c] = tp / denom
    miou = float(np.nanmean(iou_per_class))
    overall_acc = float(np.trace(cm) / max(cm.sum(), 1))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0)
    return {
        "confusion_matrix": cm,
        "iou_per_class": iou_per_class,
        "miou": miou,
        "overall_accuracy": overall_acc,
        "precision_per_class": precision,
        "recall_per_class": recall,
        "f1_per_class": f1,
        "support_per_class": support,
    }

print("compute_metrics is now defined.")

In [ ]:
# =============================================================================
# Cell 11: Load best encoder, freeze, and train a segmentation probe
# =============================================================================
# Load best encoder
ckpt = torch.load(ssl_encoder_path, map_location=device)
encoder.load_state_dict(ckpt["encoder_state_dict"])
encoder.eval()
for param in encoder.parameters():
    param.requires_grad_(False)

# Decoder head (same as UtoniaDecoderHead)
class DecoderHead(nn.Module):
    def __init__(self, num_classes, stage_channels=(64,128,256,512), dropout=0.1):
        super().__init__()
        C0, C1, C2, C3 = stage_channels
        self.fp2 = FeaturePropagation(C3 + C2, C2)
        self.fp1 = FeaturePropagation(C2 + C1, C1)
        self.fp0 = FeaturePropagation(C1 + C0, C0)
        self.head = nn.Sequential(
            nn.Linear(C0, C0 // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(C0 // 2, num_classes),
        )

    def forward(self, feats):
        d2 = self.fp2(feats["xyz2"], feats["xyz3"], feats["x2"], feats["x3"])
        d1 = self.fp1(feats["xyz1"], feats["xyz2"], feats["x1"], d2)
        d0 = self.fp0(feats["xyz0"], feats["xyz1"], feats["x0"], d1)
        logits = self.head(d0)
        logits = scatter_seq(logits, feats["order0"])
        return F.log_softmax(logits, dim=-1)

decoder = DecoderHead(NUM_CLASSES, STAGE_CHANNELS, dropout=DROPOUT).to(device)
probe_optimizer = torch.optim.AdamW(decoder.parameters(), lr=PROBE_LEARNING_RATE, weight_decay=PROBE_WEIGHT_DECAY)
probe_scheduler = torch.optim.lr_scheduler.StepLR(probe_optimizer, step_size=PROBE_LR_DECAY_STEP, gamma=PROBE_LR_DECAY_GAMMA)
nll = nn.NLLLoss(weight=class_weights_t.to(device))

probe_history = {"epoch": [], "train_loss": [], "val_loss": [], "val_miou": []}
best_val_miou = -1.0
best_probe_path = os.path.join(CHECKPOINT_DIR, "barlow_probe_best.pth")

t0 = time.time()
for epoch in range(1, PROBE_EPOCHS + 1):
    decoder.train()
    running_loss = 0.0
    for pts_t, feat_t, lbl_t in train_loader:
        pts_t, feat_t, lbl_t = pts_t.to(device), feat_t.to(device) if feat_t is not None else None, lbl_t.to(device)
        probe_optimizer.zero_grad()
        with torch.no_grad():
            feats = encoder(pts_t, feat_t, mask_bn=None)
        log_probs = decoder(feats)
        loss = nll(log_probs.reshape(-1, NUM_CLASSES), lbl_t.reshape(-1))
        loss.backward()
        probe_optimizer.step()
        running_loss += loss.item() * pts_t.size(0)
    train_loss = running_loss / len(train_ds)
    probe_scheduler.step()

    # Validation
    decoder.eval()
    val_running_loss = 0.0
    all_pred, all_true = [], []
    with torch.no_grad():
        for pts_t, feat_t, lbl_t in val_loader:
            pts_t, feat_t, lbl_t = pts_t.to(device), feat_t.to(device) if feat_t is not None else None, lbl_t.to(device)
            feats = encoder(pts_t, feat_t, mask_bn=None)
            log_probs = decoder(feats)
            loss = nll(log_probs.reshape(-1, NUM_CLASSES), lbl_t.reshape(-1))
            val_running_loss += loss.item() * pts_t.size(0)
            all_pred.append(log_probs.argmax(dim=-1).cpu().numpy().reshape(-1))
            all_true.append(lbl_t.cpu().numpy().reshape(-1))
    val_loss = val_running_loss / len(val_ds)
    val_miou = compute_metrics(np.concatenate(all_true), np.concatenate(all_pred))["miou"]
    probe_history["epoch"].append(epoch)
    probe_history["train_loss"].append(train_loss)
    probe_history["val_loss"].append(val_loss)
    probe_history["val_miou"].append(val_miou)

    if val_miou > best_val_miou:
        best_val_miou = val_miou
        torch.save({
            "decoder_state_dict": decoder.state_dict(),
            "epoch": epoch,
            "val_miou": val_miou,
            "stage_channels": STAGE_CHANNELS,
            "num_classes": NUM_CLASSES,
        }, best_probe_path)

    if epoch == 1 or epoch % 5 == 0 or epoch == PROBE_EPOCHS:
        elapsed = time.time() - t0
        print(f"Probe Epoch {epoch:3d}/{PROBE_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_mIoU={val_miou:.4f} | best={best_val_miou:.4f} | {elapsed:.1f}s")

# Load best probe
ckpt = torch.load(best_probe_path, map_location=device)
decoder.load_state_dict(ckpt["decoder_state_dict"])
decoder.eval()
print(f"Best probe val mIoU: {best_val_miou:.4f}")

# Save training history
pd.DataFrame(probe_history).to_csv(os.path.join(METRICS_DIR, "probe_train_history.csv"), index=False)

In [ ]:
# =============================================================================
# Cell 12: Utility: whole-cloud inference (same as before)
# =============================================================================
def predict_full_cloud(encoder, decoder, points_norm, features, n_points=N_POINTS_PER_SAMPLE,
                       n_repeats=N_REPEATS_EVAL, batch_size=PROBE_BATCH_SIZE, device=device):
    N = points_norm.shape[0]
    prob_sum = np.zeros((N, NUM_CLASSES), dtype=np.float64)
    count = np.zeros(N, dtype=np.int64)
    feat_full = features if (features is not None and ADDITIONAL_CHANNELS > 0) else None

    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        for rep in range(n_repeats):
            perm = np.random.default_rng(rep * 97 + 1).permutation(N)
            n_chunks = int(np.ceil(N / n_points))
            chunks = []
            for c in range(n_chunks):
                chunk_idx = perm[c * n_points:(c + 1) * n_points]
                if len(chunk_idx) < n_points:
                    pad_rng = np.random.default_rng(rep * 1000 + c)
                    pad = pad_rng.choice(perm, size=n_points - len(chunk_idx), replace=True)
                    chunk_idx = np.concatenate([chunk_idx, pad])
                chunks.append(chunk_idx)

            for b_start in range(0, len(chunks), batch_size):
                batch_chunks = chunks[b_start:b_start + batch_size]
                batch_idx = np.stack(batch_chunks)  # [B, n_points]
                pts_b = points_norm[batch_idx]     # [B, n_points, 3]
                pts_t = torch.from_numpy(pts_b.astype(np.float32)).permute(0, 2, 1).to(device)
                if feat_full is not None:
                    feat_b = feat_full[batch_idx]
                    feat_t = torch.from_numpy(feat_b.astype(np.float32)).permute(0, 2, 1).to(device)
                else:
                    feat_t = None
                feats = encoder(pts_t, feat_t, mask_bn=None)
                logits = decoder(feats)  # [B, n_points, C] log-probs
                probs = torch.exp(logits).cpu().numpy()
                for bi, chunk_idx in enumerate(batch_chunks):
                    np.add.at(prob_sum, chunk_idx, probs[bi])
                    np.add.at(count, chunk_idx, 1)

    count[count == 0] = 1
    avg_probs = prob_sum / count[:, None]
    preds = np.argmax(avg_probs, axis=1)
    mean_confidence = float(avg_probs.max(axis=1).mean())
    return preds, avg_probs, mean_confidence

def compute_metrics(y_true, y_pred, num_classes=NUM_CLASSES):
    labels = list(range(num_classes))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    iou_per_class = np.full(num_classes, np.nan)
    for c in range(num_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        denom = tp + fp + fn
        if denom > 0:
            iou_per_class[c] = tp / denom
    miou = float(np.nanmean(iou_per_class))
    overall_acc = float(np.trace(cm) / max(cm.sum(), 1))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0)
    return {
        "confusion_matrix": cm,
        "iou_per_class": iou_per_class,
        "miou": miou,
        "overall_accuracy": overall_acc,
        "precision_per_class": precision,
        "recall_per_class": recall,
        "f1_per_class": f1,
        "support_per_class": support,
    }

print("Inference utility ready.")

In [ ]:
# =============================================================================
# Cell 13: Evaluate Barlow Twins model on clean GT and on all degraded variants
# =============================================================================
# First, evaluate on GT
print("Evaluating on clean GT...")
t_start = time.time()
preds_gt, probs_gt, conf_gt = predict_full_cloud(encoder, decoder, gt_xyz_norm, gt_features)
bt_gt_metrics = compute_metrics(gt_semantic, preds_gt)
bt_gt_metrics["mean_confidence"] = conf_gt
bt_gt_metrics["inference_time_s"] = time.time() - t_start
bt_gt_metrics["n_points"] = N_GT

print(f"BT model GT mIoU: {bt_gt_metrics['miou']:.4f}")

# Save GT confusion matrix
np.save(os.path.join(CM_DIR, "BT_GT_confusion.npy"), bt_gt_metrics["confusion_matrix"])

# ---- Read manifest and evaluate all variants --------------------------------
manifest_df = pd.read_csv(MANIFEST_PATH)
variant_results = []

print(f"Evaluating on {len(manifest_df)} degraded variants from manifest...")

for idx, row in manifest_df.iterrows():
    category = row["category"]
    level = row["level"]

    # Construct paths from SOURCE_BASE_DIR (local session)
    # Expected structure: SOURCE_BASE_DIR/noisy_reconstruction/degraded/{category}/{level}/{PLANT_NAME}.ply
    variant_dir = os.path.join(SOURCE_BASE_DIR, "noisy_reconstruction", "degraded", category, str(level))
    v_ply = os.path.join(variant_dir, f"{PLANT_NAME}.ply")
    v_sem = os.path.join(variant_dir, f"{PLANT_NAME}_SemanticLabels.txt")

    if not os.path.exists(v_ply):
        print(f"Warning: {v_ply} not found, skipping.")
        continue
    if not os.path.exists(v_sem):
        print(f"Warning: {v_sem} not found, skipping.")
        continue

    print(f"Processing: {category}/{level} ...")

    v_points, v_rgb = load_point_cloud(v_ply)
    v_semantic = load_label_file(v_sem, dtype=np.int64)

    if v_points.shape[0] != v_semantic.shape[0]:
        print(f"Warning: point/label mismatch for {category}/{level}, skipping.")
        continue

    v_xyz_norm = normalize_xyz(v_points)
    v_features = make_features(v_rgb) if v_rgb is not None else None

    t0 = time.time()
    v_preds, v_probs, v_conf = predict_full_cloud(encoder, decoder, v_xyz_norm, v_features)
    inf_time = time.time() - t0
    m = compute_metrics(v_semantic, v_preds)
    delta_miou = bt_gt_metrics["miou"] - m["miou"]
    degradation_pct = 100.0 * delta_miou / bt_gt_metrics["miou"] if bt_gt_metrics["miou"] > 0 else np.nan

    result = {
        "category": category,
        "level": level,
        "gt_miou": bt_gt_metrics["miou"],
        "miou": m["miou"],
        "delta_miou": delta_miou,
        "degradation_pct": degradation_pct,
        "overall_accuracy": m["overall_accuracy"],
        "mean_confidence": v_conf,
        "inference_time_s": inf_time,
        "n_points": v_points.shape[0],
    }
    # Add per-class IoU
    for c in range(NUM_CLASSES):
        name = SEMANTIC_CLASS_NAMES.get(c, str(c)).replace(" ", "_")
        result[f"iou_{name}"] = m["iou_per_class"][c]
    variant_results.append(result)

    print(f"  {category}/{level}: mIoU={m['miou']:.4f}, Δ={delta_miou:+.4f}, deg={degradation_pct:+.1f}%")

variant_df = pd.DataFrame(variant_results)
variant_df.to_csv(os.path.join(METRICS_DIR, "bt_variant_metrics.csv"), index=False)

print(f"Evaluated {len(variant_df)} variants. Results saved to {METRICS_DIR}/bt_variant_metrics.csv")

In [ ]:
# =============================================================================
# Cell 14: Load PTv3 supervised baseline and evaluate on GT and variants
# =============================================================================
print("\nLoading PTv3 supervised baseline from existing checkpoint...")

# ---- Define PointTransformerV3Lite (matches the saved checkpoint) ----------
class PointTransformerV3Lite(nn.Module):
    """Full segmentation model with encoder + decoder head, same as in Utonia notebook."""
    def __init__(self, num_classes, additional_channel=3,
                 stage_channels=(64,128,256,512), stage_depths=(2,2,2,2),
                 num_heads=4, patch_size=32, pool_ratio=4, grid_res=1024, dropout=0.1):
        super().__init__()
        in_ch = 3 + additional_channel
        self.grid_res = grid_res
        C0, C1, C2, C3 = stage_channels
        self.input_proj = nn.Sequential(nn.Linear(in_ch, C0), nn.GELU())
        self.stage0 = SerializedAttentionStage(C0, stage_depths[0], num_heads, patch_size, dropout=dropout)
        self.pool0 = SerializedPool(C0, C1, pool_ratio)
        self.stage1 = SerializedAttentionStage(C1, stage_depths[1], num_heads, patch_size, dropout=dropout)
        self.pool1 = SerializedPool(C1, C2, pool_ratio)
        self.stage2 = SerializedAttentionStage(C2, stage_depths[2], num_heads, patch_size, dropout=dropout)
        self.pool2 = SerializedPool(C2, C3, pool_ratio)
        self.stage3 = SerializedAttentionStage(C3, stage_depths[3], num_heads, patch_size, dropout=dropout)

        self.fp2 = FeaturePropagation(C3 + C2, C2)
        self.fp1 = FeaturePropagation(C2 + C1, C1)
        self.fp0 = FeaturePropagation(C1 + C0, C0)

        self.head = nn.Sequential(
            nn.Linear(C0, C0 // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(C0 // 2, num_classes),
        )

    def forward(self, xyz_bcn, features_bcn=None):
        xyz = xyz_bcn.transpose(1, 2).contiguous()          # [B, N, 3]
        if features_bcn is not None and features_bcn.shape[1] > 0:
            feat = features_bcn.transpose(1, 2).contiguous()
            x_in = torch.cat([xyz, feat], dim=-1)
        else:
            x_in = xyz

        order0 = serialize_order(xyz, self.grid_res)
        xyz0 = gather_seq(xyz, order0)
        x_in = gather_seq(x_in, order0)

        x0 = self.input_proj(x_in)
        x0 = self.stage0(x0, xyz0)

        x1, xyz1 = self.pool0(x0, xyz0)
        x1 = self.stage1(x1, xyz1)

        x2, xyz2 = self.pool1(x1, xyz1)
        x2 = self.stage2(x2, xyz2)

        x3, xyz3 = self.pool2(x2, xyz2)
        x3 = self.stage3(x3, xyz3)

        d2 = self.fp2(xyz2, xyz3, x2, x3)
        d1 = self.fp1(xyz1, xyz2, x1, d2)
        d0 = self.fp0(xyz0, xyz1, x0, d1)

        logits = self.head(d0)
        logits = scatter_seq(logits, order0)
        return F.log_softmax(logits, dim=-1)

if os.path.exists(PTV3_CKPT_PATH):
    ckpt = torch.load(PTV3_CKPT_PATH, map_location=device)
    # Use hyperparameters saved in checkpoint if available, else defaults
    ptv3_model = PointTransformerV3Lite(
        num_classes=ckpt.get("num_classes", NUM_CLASSES),
        additional_channel=ckpt.get("additional_channel", ADDITIONAL_CHANNELS),
        stage_channels=tuple(ckpt.get("stage_channels", STAGE_CHANNELS)),
        stage_depths=tuple(ckpt.get("stage_depths", STAGE_DEPTHS)),
        num_heads=ckpt.get("num_heads", NUM_HEADS),
        patch_size=ckpt.get("patch_size", PATCH_SIZE),
        pool_ratio=ckpt.get("pool_ratio", POOL_RATIO),
        grid_res=ckpt.get("grid_res", GRID_RES),
        dropout=DROPOUT,
    ).to(device)
    ptv3_model.load_state_dict(ckpt["model_state_dict"])
    ptv3_model.eval()
    print("PTv3 model loaded successfully.")

    # Define inference function for PTv3 (similar to predict_full_cloud but using PTv3 model)
    def predict_full_cloud_ptv3(model, points_norm, features, n_points=N_POINTS_PER_SAMPLE,
                                n_repeats=N_REPEATS_EVAL, batch_size=PROBE_BATCH_SIZE, device=device):
        N = points_norm.shape[0]
        prob_sum = np.zeros((N, NUM_CLASSES), dtype=np.float64)
        count = np.zeros(N, dtype=np.int64)
        feat_full = features if (features is not None and ADDITIONAL_CHANNELS > 0) else None
        model.eval()
        with torch.no_grad():
            for rep in range(n_repeats):
                perm = np.random.default_rng(rep * 97 + 1).permutation(N)
                n_chunks = int(np.ceil(N / n_points))
                chunks = []
                for c in range(n_chunks):
                    chunk_idx = perm[c * n_points:(c + 1) * n_points]
                    if len(chunk_idx) < n_points:
                        pad_rng = np.random.default_rng(rep * 1000 + c)
                        pad = pad_rng.choice(perm, size=n_points - len(chunk_idx), replace=True)
                        chunk_idx = np.concatenate([chunk_idx, pad])
                    chunks.append(chunk_idx)
                for b_start in range(0, len(chunks), batch_size):
                    batch_chunks = chunks[b_start:b_start + batch_size]
                    batch_idx = np.stack(batch_chunks)
                    pts_b = points_norm[batch_idx]
                    pts_t = torch.from_numpy(pts_b.astype(np.float32)).permute(0,2,1).to(device)
                    if feat_full is not None:
                        feat_b = feat_full[batch_idx]
                        feat_t = torch.from_numpy(feat_b.astype(np.float32)).permute(0,2,1).to(device)
                    else:
                        feat_t = None
                    logits = model(pts_t, feat_t)
                    probs = torch.exp(logits).cpu().numpy()
                    for bi, chunk_idx in enumerate(batch_chunks):
                        np.add.at(prob_sum, chunk_idx, probs[bi])
                        np.add.at(count, chunk_idx, 1)
        count[count == 0] = 1
        avg_probs = prob_sum / count[:, None]
        preds = np.argmax(avg_probs, axis=1)
        mean_conf = float(avg_probs.max(axis=1).mean())
        return preds, avg_probs, mean_conf

    # Evaluate PTv3 on GT
    print("Evaluating PTv3 on clean GT...")
    preds_ptv3_gt, _, conf_ptv3_gt = predict_full_cloud_ptv3(ptv3_model, gt_xyz_norm, gt_features)
    ptv3_gt_metrics = compute_metrics(gt_semantic, preds_ptv3_gt)
    ptv3_gt_metrics["mean_confidence"] = conf_ptv3_gt
    np.save(os.path.join(CM_DIR, "PTv3_GT_confusion.npy"), ptv3_gt_metrics["confusion_matrix"])
    print(f"PTv3 GT mIoU: {ptv3_gt_metrics['miou']:.4f}")

    # Evaluate PTv3 on variants (using SOURCE_BASE_DIR paths)
    ptv3_variant_rows = []
    print("Evaluating PTv3 on degraded variants...")
    for _, row in manifest_df.iterrows():
        category, level = row["category"], row["level"]
        # Construct paths from SOURCE_BASE_DIR
        variant_dir = os.path.join(SOURCE_BASE_DIR, "noisy_reconstruction", "degraded", category, str(level))
        v_ply = os.path.join(variant_dir, f"{PLANT_NAME}.ply")
        v_sem = os.path.join(variant_dir, f"{PLANT_NAME}_SemanticLabels.txt")
        if not os.path.exists(v_ply) or not os.path.exists(v_sem):
            print(f"Warning: {category}/{level} files not found, skipping.")
            continue
        v_points, v_rgb = load_point_cloud(v_ply)
        v_semantic = load_label_file(v_sem, dtype=np.int64)
        if v_points.shape[0] != v_semantic.shape[0]:
            print(f"Warning: point/label mismatch for {category}/{level}, skipping.")
            continue
        v_xyz_norm = normalize_xyz(v_points)
        v_features = make_features(v_rgb) if v_rgb is not None else None
        v_preds, _, v_conf = predict_full_cloud_ptv3(ptv3_model, v_xyz_norm, v_features)
        m = compute_metrics(v_semantic, v_preds)
        delta = ptv3_gt_metrics["miou"] - m["miou"]
        deg = 100.0 * delta / ptv3_gt_metrics["miou"] if ptv3_gt_metrics["miou"] > 0 else np.nan
        row_dict = {
            "category": category,
            "level": level,
            "gt_miou": ptv3_gt_metrics["miou"],
            "miou": m["miou"],
            "delta_miou": delta,
            "degradation_pct": deg,
        }
        ptv3_variant_rows.append(row_dict)
        print(f"  {category}/{level}: mIoU={m['miou']:.4f}, Δ={delta:+.4f}, deg={deg:+.1f}%")
    ptv3_variant_df = pd.DataFrame(ptv3_variant_rows)
    ptv3_variant_df.to_csv(os.path.join(METRICS_DIR, "ptv3_variant_metrics.csv"), index=False)
    print(f"PTv3 evaluation done. Saved {len(ptv3_variant_df)} variants.")
else:
    print("PTv3 checkpoint not found; skipping baseline comparison.")
    ptv3_variant_df = None

In [ ]:
# =============================================================================
# Cell 15: Generate comprehensive figures and summary tables
# =============================================================================
# (We'll create: 1) SSL loss curve, 2) Probe training curves, 3) GT confusion matrices side-by-side,
#  4) mIoU vs severity per category (both models), 5) Degradation heatmaps, 6) Per-class IoU degradation,
#  7) Robustness gap chart, 8) Summary table with ΔmIoU and degradation%.)

# Plotting setup
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 220, "font.size": 10})

# 1) SSL loss
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(ssl_history["epoch"], ssl_history["loss"], color='#1a237e')
ax.set_xlabel("Epoch"); ax.set_ylabel("Barlow Twins loss")
ax.set_title("SSL Pretraining Loss")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "ssl_pretrain_loss.png"))
plt.close(fig)

# 2) Probe training
fig, ax1 = plt.subplots(figsize=(8,4))
ax1.plot(probe_history["epoch"], probe_history["train_loss"], label="Train loss", color='blue')
ax1.plot(probe_history["epoch"], probe_history["val_loss"], label="Val loss", color='red')
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend(loc='upper left')
ax2 = ax1.twinx()
ax2.plot(probe_history["epoch"], probe_history["val_miou"], label="Val mIoU", color='green')
ax2.set_ylabel("mIoU")
ax2.legend(loc='upper right')
ax1.set_title("Linear Probe Training")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "probe_training.png"))
plt.close(fig)

# 3) GT confusion matrices (BT vs PTv3 if available)
fig, axes = plt.subplots(1, 2, figsize=(10,4))
cm_bt = bt_gt_metrics["confusion_matrix"] / bt_gt_metrics["confusion_matrix"].sum(axis=1, keepdims=True)
sns.heatmap(cm_bt, annot=True, fmt=".2f", xticklabels=list(SEMANTIC_CLASS_NAMES.values()),
            yticklabels=list(SEMANTIC_CLASS_NAMES.values()), ax=axes[0], cmap='Blues')
axes[0].set_title("BT model GT")
if ptv3_gt_metrics is not None:
    cm_pt = ptv3_gt_metrics["confusion_matrix"] / ptv3_gt_metrics["confusion_matrix"].sum(axis=1, keepdims=True)
    sns.heatmap(cm_pt, annot=True, fmt=".2f", xticklabels=list(SEMANTIC_CLASS_NAMES.values()),
                yticklabels=list(SEMANTIC_CLASS_NAMES.values()), ax=axes[1], cmap='Blues')
    axes[1].set_title("PTv3 GT")
fig.suptitle("Confusion Matrices on Clean GT")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "gt_confusion_comparison.png"))
plt.close(fig)

# 4) mIoU vs severity per category (for BT and PTv3)
categories = variant_df["category"].unique()
fig, axes = plt.subplots(2, 3, figsize=(15,10))
axes = axes.flatten()
for idx, cat in enumerate(categories):
    ax = axes[idx]
    sub_bt = variant_df[variant_df["category"]==cat]
    # order levels
    levels = sub_bt["level"].tolist()
    miou_bt = [bt_gt_metrics["miou"]] + sub_bt["miou"].tolist()
    labels = ["GT"] + levels
    ax.plot(labels, miou_bt, marker='o', label='Barlow Twins', color='purple')
    if ptv3_variant_df is not None:
        sub_pt = ptv3_variant_df[ptv3_variant_df["category"]==cat]
        miou_pt = [ptv3_gt_metrics["miou"]] + sub_pt["miou"].tolist()
        ax.plot(labels, miou_pt, marker='s', label='PTv3', color='orange')
    ax.set_title(cat.replace("_"," ").title())
    ax.set_ylabel("mIoU")
    ax.set_ylim(0,1)
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xticklabels(labels, rotation=15, ha='right')
for j in range(len(categories), len(axes)):
    axes[j].axis('off')
fig.suptitle("mIoU vs Severity per Degradation Category")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "miou_by_category.png"))
plt.close(fig)

# 5) Degradation heatmaps (for BT)
pivot_bt = variant_df.pivot(index="category", columns="level", values="degradation_pct")
fig, ax = plt.subplots(figsize=(8,4))
sns.heatmap(pivot_bt, annot=True, fmt=".1f", cmap="Reds", cbar=True, linewidths=0.5, ax=ax)
ax.set_title("BT model: Degradation % (mIoU drop)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "bt_degradation_heatmap.png"))
plt.close(fig)

# 6) Per-class IoU degradation (average over variants)
class_iou_cols = [f"iou_{SEMANTIC_CLASS_NAMES.get(c, str(c)).replace(' ', '_')}" for c in range(NUM_CLASSES)]
avg_iou_bt = [variant_df[col].mean() for col in class_iou_cols]
fig, ax = plt.subplots(figsize=(8,5))
x = np.arange(NUM_CLASSES)
width = 0.35
ax.bar(x - width/2, bt_gt_metrics["iou_per_class"], width, label='GT', color='purple')
ax.bar(x + width/2, avg_iou_bt, width, label='Avg over noisy', color='lightcoral')
ax.set_xticks(x); ax.set_xticklabels([SEMANTIC_CLASS_NAMES.get(c, str(c)) for c in range(NUM_CLASSES)], rotation=15)
ax.set_ylabel("IoU"); ax.set_ylim(0,1)
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("BT model: GT vs average noisy IoU per class")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "bt_per_class_iou.png"))
plt.close(fig)

# 7) Robustness gap between BT and PTv3 (if PTv3 available)
if ptv3_variant_df is not None:
    gap_df = pd.merge(variant_df[["category","level","degradation_pct"]],
                      ptv3_variant_df[["category","level","degradation_pct"]],
                      on=["category","level"], suffixes=("_bt","_ptv3"))
    gap_df["gap"] = gap_df["degradation_pct_bt"] - gap_df["degradation_pct_ptv3"]
    # Aggregate per category
    gap_cat = gap_df.groupby("category")["gap"].mean().reset_index()
    fig, ax = plt.subplots(figsize=(8,4))
    colors = ['green' if g < 0 else 'red' for g in gap_cat["gap"]]
    ax.bar(gap_cat["category"], gap_cat["gap"], color=colors)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel("BT degradation % - PTv3 degradation %")
    ax.set_title("Robustness Gap (negative = BT more robust)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "robustness_gap.png"))
    plt.close(fig)

# 8) Summary degradation table (both models)
summary_rows = []
for _, row in variant_df.iterrows():
    summary_rows.append({
        "Model": "Barlow Twins",
        "Category": row["category"],
        "Level": row["level"],
        "GT mIoU": row["gt_miou"],
        "Noisy mIoU": row["miou"],
        "ΔmIoU": row["delta_miou"],
        "Degradation %": row["degradation_pct"],
    })
if ptv3_variant_df is not None:
    for _, row in ptv3_variant_df.iterrows():
        summary_rows.append({
            "Model": "PTv3 (supervised)",
            "Category": row["category"],
            "Level": row["level"],
            "GT mIoU": row["gt_miou"],
            "Noisy mIoU": row["miou"],
            "ΔmIoU": row["delta_miou"],
            "Degradation %": row["degradation_pct"],
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(METRICS_DIR, "summary_degradation_table.csv"), index=False)
print("Summary table saved.")


In [ ]:
# =============================================================================
# Cell 16: Auto-generated Markdown report
# =============================================================================
report_lines = [
    f"# Barlow Twins SSL for Robustness — {PLANT_NAME}",
    f"- SSL pretraining epochs: {SSL_EPOCHS}, loss: {ssl_history['loss'][-1]:.6f}",
    f"- Linear probe epochs: {PROBE_EPOCHS}, best val mIoU: {best_val_miou:.4f}",
    f"- GT mIoU (BT model): {bt_gt_metrics['miou']:.4f}",
]
if ptv3_gt_metrics is not None:
    report_lines.append(f"- GT mIoU (PTv3 baseline): {ptv3_gt_metrics['miou']:.4f}")
    report_lines.append(f"- Improvement over PTv3 on GT: {bt_gt_metrics['miou'] - ptv3_gt_metrics['miou']:+.4f}")
report_lines.append("\n## Degradation Summary (mean ΔmIoU per category)")
for cat in categories:
    sub = variant_df[variant_df["category"]==cat]
    mean_delta = sub["delta_miou"].mean()
    report_lines.append(f"- {cat}: mean ΔmIoU = {mean_delta:+.4f}")
if ptv3_variant_df is not None:
    report_lines.append("\n## Robustness Gap (BT - PTv3) per category")
    for cat in categories:
        bt_deg = variant_df[variant_df["category"]==cat]["degradation_pct"].mean()
        pt_deg = ptv3_variant_df[ptv3_variant_df["category"]==cat]["degradation_pct"].mean()
        gap = bt_deg - pt_deg
        report_lines.append(f"- {cat}: {gap:+.2f}% (negative = BT more robust)")
report_lines.append("\n## Figures")
report_lines.extend([f"- {f}" for f in os.listdir(FIG_DIR) if f.endswith('.png')])
with open(os.path.join(REPORT_DIR, "barlow_twins_report.md"), "w") as f:
    f.write("\n".join(report_lines))
print("Report saved.")

# =============================================================================
print("\n✅ All done! Results saved in:", OUTPUT_DIR)